# 🔫 ARGUS Sentinel — Weapon Detection Training
**YOLOv8 custom training on 9 weapon classes**

| Class | Examples |
|---|---|
| Automatic Rifle | AK-47, M16 |
| Bazooka | RPG, launcher |
| Grenade Launcher | M79 |
| Handgun | Pistol, revolver |
| Knife | Combat knife |
| SMG | MP5, Uzi |
| Shotgun | Pump-action |
| Sniper | Rifle with scope |
| Sword | Bladed weapon |

> ⚠️ **Google Colab users**: Upload your `weapon_detection/` folder to Colab or mount Google Drive for GPU training (~15 mins vs 4 hours on CPU)

## Cell 1 — Install & Import

In [ ]:
# Install ultralytics if not already installed
try:
    import ultralytics
    print(f"✅ ultralytics {ultralytics.__version__} already installed")
except ImportError:
    print("Installing ultralytics...")
    import subprocess
    subprocess.run(["pip", "install", "ultralytics"], check=True)
    import ultralytics
    print(f"✅ Installed ultralytics {ultralytics.__version__}")

from ultralytics import YOLO
import os, torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
else:
    print("⚠ No GPU found — training on CPU (will be slow)")

## Cell 2 — Dataset Check

In [ ]:
import os

# Adjust this path if running from notebooks/ subfolder
BASE_DIR     = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
DATASET_YAML = os.path.join(BASE_DIR, "weapon_detection", "dataset.yaml")

print(f"Working directory : {BASE_DIR}")
print(f"Dataset YAML      : {DATASET_YAML}")
print(f"YAML exists       : {os.path.exists(DATASET_YAML)}")

# Count images
for split in ["train", "val"]:
    img_dir = os.path.join(BASE_DIR, "weapon_detection", split, "images")
    lbl_dir = os.path.join(BASE_DIR, "weapon_detection", split, "labels")
    imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
    print(f"  {split:5s}/images → {imgs} images | {split}/labels → {lbls} labels")

# Show weapon classes
img_dir = os.path.join(BASE_DIR, "weapon_detection", "train", "images")
classes = {}
for f in os.listdir(img_dir):
    if f.lower().endswith(('.jpg','.jpeg','.png')):
        cls = f.rsplit('_', 1)[0]
        classes[cls] = classes.get(cls, 0) + 1

print("\nWeapon classes in training set:")
for cls, cnt in sorted(classes.items()):
    print(f"  {cls:<25} {cnt} images")

## Cell 3 — Training Config

In [ ]:
import torch

# ── ADJUST THESE SETTINGS ─────────────────────────────────────
BASE_MODEL   = "yolov8n.pt"   # yolov8n=fast, yolov8s=balanced, yolov8m=accurate
EPOCHS       = 50             # 50 for demo, 100 for production
IMG_SIZE     = 640
BATCH_SIZE   = 8              # lower to 4 if you get memory errors
DEVICE       = 0 if torch.cuda.is_available() else "cpu"  # auto GPU/CPU
PROJECT_NAME = os.path.join(BASE_DIR, "runs", "detect")
RUN_NAME     = "weapons"
# ──────────────────────────────────────────────────────────────

print("Training Configuration:")
print(f"  Base model  : {BASE_MODEL}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Image size  : {IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Device      : {'GPU ✅' if DEVICE == 0 else 'CPU ⚠ (slow)'}")
print(f"  Output dir  : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")

## Cell 4 — 🚀 Start Training
> This cell will take **15 mins (GPU)** or **2–4 hours (CPU)**  
> Training progress is shown below in real time

In [ ]:
# Load base YOLOv8 model
model = YOLO(BASE_MODEL)

# Train!
results = model.train(
    data     = DATASET_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    device   = DEVICE,
    project  = PROJECT_NAME,
    name     = RUN_NAME,
    patience = 10,
    workers  = 2,
    verbose  = True,
)

print("\n✅ Training complete!")

## Cell 5 — Validate the Model

In [ ]:
best_weights = os.path.join(PROJECT_NAME, RUN_NAME, "weights", "best.pt")
print(f"Loading best weights: {best_weights}")

model = YOLO(best_weights)
metrics = model.val(data=DATASET_YAML, imgsz=IMG_SIZE)

print(f"\n── Validation Results ──────────────")
print(f"  mAP@50     : {metrics.box.map50:.3f}")
print(f"  mAP@50-95  : {metrics.box.map:.3f}")
print(f"  Precision  : {metrics.box.mp:.3f}")
print(f"  Recall     : {metrics.box.mr:.3f}")

## Cell 6 — Test on a Sample Image

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# Pick a sample image from val set
val_imgs = glob.glob(os.path.join(BASE_DIR, "weapon_detection", "val", "images", "*.jpg"))
val_imgs += glob.glob(os.path.join(BASE_DIR, "weapon_detection", "val", "images", "*.jpeg"))

if val_imgs:
    sample = val_imgs[0]
    print(f"Running inference on: {os.path.basename(sample)}")

    results = model.predict(source=sample, conf=0.4, save=True,
                            project=PROJECT_NAME, name="test_inference")

    # Show the result
    result_img = glob.glob(os.path.join(PROJECT_NAME, "test_inference", "*.jpg"))
    if result_img:
        display(IPImage(filename=result_img[0], width=600))
    
    for r in results:
        print(f"Detections: {len(r.boxes)}")
        for box in r.boxes:
            cls_id = int(box.cls)
            conf   = float(box.conf)
            names  = ["Automatic Rifle","Bazooka","Grenade Launcher","Handgun",
                      "Knife","SMG","Shotgun","Sniper","Sword"]
            print(f"  → {names[cls_id]:<20} confidence: {conf:.2%}")
else:
    print("No val images found")

## Cell 7 — Save Best Weights Path
After this notebook completes, run `python main.py` — the system will use the new weapon model.

In [ ]:
best_weights = os.path.join(PROJECT_NAME, RUN_NAME, "weights", "best.pt")

print("=" * 55)
print("  ARGUS Sentinel — Weapon Training Complete")
print("=" * 55)
print(f"  Best model saved to:")
print(f"  {best_weights}")
print()
print("  Next steps:")
print("  1. This model detects 9 weapon classes")
print("  2. Tell Antigravity to integrate it into yolo.py")
print("  3. Run python main.py to start the full system")
print("=" * 55)